# Station Status Fact Data - Gold Layer

## Objective
Extract, transform, and measure time-series station operational metrics from silver.silver_velib to build a standardized Station Status Fact Gold Delta table (gold.gold_fact_status).

## Data Flow
silver.silver_velib → Spark SQL / DataFrame → gold.gold_fact_status

## Source
The underlying data comes from the Paris OpenData API: Vélib' - Emplacements des stations.

## Input
Silver Delta table: silver.silver_velib

## Output
Gold Delta table: gold.gold_fact_status

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates transactional events, bike availability counts, dock occupancy, and temporal status metrics to provide a central fact table for downstream business intelligence and reporting.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_velib into PySpark and display initial dataset.
2. **Extract Fact Metrics:** Query station IDs, timestamps, bike counts, free docks, and availability status.
3. **Write to Gold:** Persist transactional status records to gold.gold_fact_status Delta table.

In [0]:
# Load data from silver schema
df_silver_velib=spark.table('workspace.silver.silver_velib')

In [0]:
# display the dataframe 
df_silver_velib.display()

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract Fact Station Status Data
query_fact_status = """
SELECT 
    stationcode AS station_code,
    CAST(duedate AS TIMESTAMP) AS snapshot_time,
    is_installed,
    is_renting,
    is_returning,
    numbikesavailable AS bikes_available,
    mechanical AS mechanical_bikes,
    ebike AS electric_bikes,
    numdocksavailable AS docks_available,
    ROUND(
        numbikesavailable / CASE WHEN capacity = 0 THEN 1 ELSE capacity END, 
        4
    ) AS occupancy_rate
FROM silver.silver_velib
"""

df_fact_status = spark.sql(query_fact_status)

In [0]:
# display df_fact_status
df_fact_status.display()

# WRITING GOLD TABLE

In [0]:
# writing df_fact_status to gold schema
df_fact_status\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_fact_status")

# CHECKING THE GOLD TABLE

In [0]:
%sql
SELECT * 
FROM workspace.gold.gold_fact_status;